# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Casillas leidas de la imagen (fila, columna)
        self.start = (4, 0)
        self.walls = {(1, 1), (1, 2), (2, 4), (3, 2)}
        self.slippery_states = {(2, 3), (3, 3), (2, 5)}

        self.terminal_states = {
            (0, 5): 10.0,    # entrega
            (4, 5): 2.0,     # carga
            (0, 3): -10.0,   # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (2, 2): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height)
                for c in range(self.width)
                if self.is_valid_state((r, c))]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...].
        Las probabilidades dependen de si 'state' es resbaloso.
        Si golpea pared/borde -> se queda en 'state'.
        """
        # un terminal es absorbente
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_main, p_side = 0.60, 0.20
        else:
            p_main, p_side = 0.90, 0.05

        dr, dc = action
        # desviaciones perpendiculares a la accion elegida
        left  = (-dc,  dr)
        right = ( dc, -dr)

        probs = {}
        for move, p in [(action, p_main), (left, p_side), (right, p_side)]:
            r, c = state
            nxt = (r + move[0], c + move[1])
            if not self.is_valid_state(nxt):
                nxt = state
            probs[nxt] = probs.get(nxt, 0.0) + p

        return list(probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    total = 0.0
    for next_state, p in grid.get_transition_probs(state, action):
        total += p * V[next_state]
    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for i in range(1, max_iter + 1):
        delta = 0.0
        for s in grid.states():
            if grid.is_terminal(s):
                new_v = grid.get_reward(s)
            else:
                best = max(expected_next_value(grid, s, a, V)
                           for a in grid.actions)
                new_v = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(new_v - V[s]))
            V[s] = new_v
        if delta < threshold:
            return V, i
    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(grid.actions,
                        key=lambda a: expected_next_value(grid, s, a, V))
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for _ in range(max_iter):
        delta = 0.0
        for s in grid.states():
            if grid.is_terminal(s):
                new_v = grid.get_reward(s)
            else:
                new_v = grid.get_reward(s) + grid.gamma * \
                    expected_next_value(grid, s, policy[s], V)
            delta = max(delta, abs(new_v - V[s]))
            V[s] = new_v
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. politica inicial arbitraria (todo hacia arriba)
    policy = {s: (-1, 0) for s in grid.states() if not grid.is_terminal(s)}
    history = []

    V = None
    for _ in range(max_iter):
        # 2. evaluacion
        V = policy_evaluation(grid, policy, threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        # 4. contar cambios y repetir hasta estabilidad
        changed = sum(1 for s in policy if policy[s] != new_policy[s])
        history.append(changed)
        policy = new_policy
        if changed == 0:
            break

    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 19

Valores:
 -5.794 |  -6.256 |  -6.668 | -10.000 |  +7.607 | +10.000
 -5.249 |   WALL   |   WALL   |  +1.468 |  +3.670 |  +7.607
 -4.663 |  -4.110 |  -4.377 |  -1.214 |   WALL   |  +4.856
 -4.035 |  -3.337 |   WALL   |  -0.404 |  +1.659 |  +3.150
 -3.337 |  -2.476 |  -1.499 |  -0.449 |  +0.727 |  +2.000

Política:
 ↓  |  ←  |  ←  | -10 |  →  | +10
 ↓  |  #  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  →  |  ↑  |  #  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 →  |  →  |  →  |  →  |  →  | +2

=== POLICY ITERATION ===
Historia: [18, 7, 5, 4, 0]

Valores:
 -5.794 |  -6.256 |  -6.668 | -10.000 |  +7.607 | +10.000
 -5.249 |   WALL   |   WALL   |  +1.468 |  +3.670 |  +7.607
 -4.663 |  -4.110 |  -4.377 |  -1.214 |   WALL   |  +4.856
 -4.035 |  -3.337 |   WALL   |  -0.404 |  +1.659 |  +3.150
 -3.337 |  -2.476 |  -1.499 |  -0.449 |  +0.727 |  +2.000

Política:
 ↓  |  ←  |  ←  | -10 |  →  | +10
 ↓  |  #  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  →  |  ↑  |  #  |  ↑ 
 ↓  | 


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
